In [3]:
import pandas as pd
import numpy as np
import os
from matplotlib import pyplot as plt
from scipy.interpolate import BSpline
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix
from skfda.representation.grid import FDataGrid
from skfda.preprocessing.dim_reduction.feature_extraction import FPCA


# Olive Oil Classification by Signal

---

The dataset contains **60 authenticated samples** of extra virgin olive oils originating from four European producing countries:

| Class | Country |
|---|---|
| 1 | Greece |
| 2 | Italy |
| 3 | Portugal |
| 4 | Spain |

Each sample is represented as a **spectral signal**. The goal is to classify the country of origin from the signal alone.

We compare two pipelines:

| Method | Feature Extraction | Classifier |
|---|---|---|
| **B-Spline + SVM** | Cubic B-spline basis coefficients | SVM |
| **FPCA + SVM** | Functional principal component scores | SVM |

---


## 1. Load & Prepare Data

---

The last column of `OliveOilSignals.csv` contains class labels (country of origin). Samples are split **50/50** into train and test sets, with each split z-score normalised independently.

| Split | Samples |
|---|---|
| Train | 1 – 30 |
| Test | 31 – 60 |


In [4]:
olive = pd.read_csv('Question4.csv', header=None)

X = np.linspace(0, 1, olive.shape[1]-1)  # x-axis grid, excluding label column
Y = olive.iloc[:, -1]

# Setting first 30 as training data
trainx = olive.iloc[:30, :-1]
trainx = (trainx - trainx.mean()) / trainx.std()
trainy = Y[:30]

testx = olive.iloc[30:, :-1]
testx = (testx - testx.mean()) / testx.std()
testy = Y[30:]


---

## 2. Method 1 — Cubic B-Spline + SVM

---

We represent each signal as a linear combination of **cubic B-spline basis functions**. The least-squares coefficients of this fit become the feature vector passed to the SVM.

$$f(x) = \sum_{j=1}^{K} c_j B_j(x)$$

**Basis parameters:**

| Parameter | Value |
|---|---|
| Order $M$ | 4 (cubic) |
| Interior knots | 70, uniform on $[0,1]$ |
| Degrees of freedom | $K = n\_knots + M = 74$ |


### 2.1 Construct spline basis


In [5]:
M = 4  # order

n_knots = 70
knots = np.linspace(0, 1, num=n_knots)

knot_low = X[0]
knot_high = X[-1]

# M-1 converts from order to degree
augmented_knots = np.append(np.append([knot_low]*(M-1), knots), [knot_high]*(M-1))

DOF = n_knots + M

spline = BSpline(knots, np.eye(DOF), M-1, extrapolate=True)
B = spline(X)[:, :-2]


### 2.2 Fit SVM on spline coefficients


In [6]:
Bcoef_train = np.linalg.lstsq(B, trainx.T)[0].T

svm_clf = SVC()
svm_clf.fit(Bcoef_train, trainy)

Bcoef_test = np.linalg.lstsq(B, testx.T)[0].T

pred = svm_clf.predict(Bcoef_test)

accuracy = sum(pred == testy) / len(testy)
print("Accuracy:", accuracy)

conf = confusion_matrix(testy, pred)
conf = pd.DataFrame(conf,
                    index=['Class 1', 'Class 2', 'Class 3', 'Class 4'],
                    columns=['Predicted 1', 'Predicted 2', 'Predicted 3', 'Predicted 4'])
print('Confusion Matrix - Splines\n', conf)
print("Accuracy:", accuracy)


Accuracy: 0.8666666666666667
Confusion Matrix - Splines
          Predicted 1  Predicted 2  Predicted 3  Predicted 4
Class 1            5            0            0            0
Class 2            0            9            0            0
Class 3            0            1            1            2
Class 4            0            1            0           11
Accuracy: 0.8666666666666667


---

## 3. Method 2 — Functional PCA + SVM

---

**Functional PCA (FPCA)** generalises classical PCA to function-valued data. Each signal is projected onto the leading functional principal components, and the resulting scores are used as features for the SVM.

$$x_i(t) \approx \mu(t) + \sum_{k=1}^{K} \xi_{ik} \, \phi_k(t)$$

where $\phi_k$ are the functional eigenfunctions and $\xi_{ik}$ are the PC scores used as features.

We sweep over $K \in \{2, 5, 8, 10\}$ components to find the optimal dimensionality.


### 3.1 Prepare functional data object


In [7]:
X = olive.iloc[:, :-1]
X = (X - X.mean()) / X.std()
X = FDataGrid(X)
Y = olive.iloc[:, -1]


### 3.2 Fit SVM on FPCA features

FPCA is fit on the full dataset and the resulting scores are split into train/test sets.


In [8]:
svm_clf = SVC()

# Determine best number of harmonics for FPCA
for num_components in [2, 5, 8, 10]:

    # Get train and test data in same feature space
    fpca = FPCA(n_components=num_components)
    x_fpca = fpca.fit_transform(X)

    # Split for train/test
    trainy = Y[:30]
    testy = Y[30:]
    trainx = x_fpca[:30]
    testx = x_fpca[30:]

    # Fit model
    svm_clf.fit(trainx, trainy)
    yhat = svm_clf.predict(testx)

    # Get metrics and visualize
    accuracy = sum(yhat == testy) / len(testy)

    conf = confusion_matrix(testy, yhat)
    conf = pd.DataFrame(conf,
                        index=['Class 1', 'Class 2', 'Class 3', 'Class 4'],
                        columns=['Predicted 1', 'Predicted 2', 'Predicted 3', 'Predicted 4'])
    print(f"Number of Components= {num_components}")
    print('Confusion Matrix - FPCA\n', conf)
    print("Accuracy:", accuracy)
    print("---------------------------------------------------------------", "\n")


Number of Components= 2
Confusion Matrix - FPCA
          Predicted 1  Predicted 2  Predicted 3  Predicted 4
Class 1            4            0            0            1
Class 2            0            9            0            0
Class 3            1            1            1            1
Class 4            0            1            0           11
Accuracy: 0.8333333333333334
--------------------------------------------------------------- 

Number of Components= 5
Confusion Matrix - FPCA
          Predicted 1  Predicted 2  Predicted 3  Predicted 4
Class 1            5            0            0            0
Class 2            0            9            0            0
Class 3            0            1            1            2
Class 4            0            1            0           11
Accuracy: 0.8666666666666667
--------------------------------------------------------------- 

Number of Components= 8
Confusion Matrix - FPCA
          Predicted 1  Predicted 2  Predicted 3  Predicted 4
Cla

---

## 4. Results Summary

---

| Method | # Features | Test Accuracy |
|---|---|---|
| B-Spline + SVM | 74 (spline coefficients) | **86.7%** |
| FPCA + SVM | 2 components | 83.3% |
| FPCA + SVM | 5 components | **86.7%** |
| FPCA + SVM | 8 components | 83.3% |
| FPCA + SVM | 10 components | 83.3% |

> **Key finding:** FPCA with just 5 components matches the B-Spline SVM accuracy, using far fewer features — demonstrating that the signals contain most of their class-discriminating information in a low-dimensional functional subspace.

---
